In [2]:
import torch
import torch.nn as nn
from monai.losses import DiceLoss

/work/scratch/sanyal/miniconda3/envs/bratseg/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
x = torch.rand((1,1,4,4))
p = torch.nn.ConstantPad2d(1, 999)(x)
y = torch.where(p == 999, torch.randn(p.shape), p)
fg_mask = torch.where(p == 999, torch.zeros(p.shape), torch.ones(p.shape))

In [21]:
x

tensor([[[[0.8626, 0.9795, 0.6178, 0.2714],
          [0.3298, 0.2334, 0.3553, 0.5714],
          [0.4060, 0.6808, 0.6350, 0.3771],
          [0.8337, 0.8971, 0.3572, 0.0365]]]])

In [24]:
fg_mask

tensor([[[[0., 0., 0., 0., 0., 0.],
          [0., 1., 1., 1., 1., 0.],
          [0., 1., 1., 1., 1., 0.],
          [0., 1., 1., 1., 1., 0.],
          [0., 1., 1., 1., 1., 0.],
          [0., 0., 0., 0., 0., 0.]]]])

In [3]:
def complement_randomly_chosen_elements(y_true, frac=0.3):
    assert ((y_true >= 0) & (y_true <= 1)).all()
    #create y_pred by randomly swapping x% of y_true values
    y_pred = y_true.detach().clone()
    num_elements = y_true.numel()
    num_samples = int(num_elements * frac)  # x% of the elements
    #swap randomly chosen indices
    idxs = torch.randperm(num_elements)[:num_samples]
    idxs = torch.unravel_index(idxs,shape=y_true.shape)
    y_pred[idxs] = 1 - y_pred[idxs]
    return y_pred

In [4]:
def logit(x, epsilon=1e-7):
    """
    Computes the inverse of the sigmoid function (logit) in a numerically stable way.
    
    Args:
        x (torch.Tensor): Input tensor with values in (0, 1).
        epsilon (float): Small value to clip inputs to avoid inf/nan.
        
    Returns:
        torch.Tensor: Logit values.
    """
    # Clip x to avoid values exactly 0 or 1
    x = torch.clamp(x, epsilon, 1 - epsilon)
    return torch.log(x) - torch.log(1 - x)

# Generate samples strictly inside (0, 1)
def sample_strictly_in_0_1(size):
    epsilon = 1e-4  # Small value to avoid boundaries
    return torch.rand(size) * (1 - 2 * epsilon) + epsilon

In [5]:
shape = (4,3,96,96,96)
y_true = torch.rand(shape)
y_pred = complement_randomly_chosen_elements(y_true)
#pad tensors
y_true = torch.nn.ConstantPad3d(16, 0.0)(y_true)
y_true = y_true.gt(0.7)
p = torch.nn.ConstantPad3d(16, 999)(y_pred)
y_pred = logit(torch.where(p == 999, torch.rand(p.shape), p))
fg_mask = torch.where(p == 999, torch.zeros(p.shape), torch.ones(p.shape))

print(y_true.shape)
print(y_pred.shape)

torch.Size([4, 3, 128, 128, 128])
torch.Size([4, 3, 128, 128, 128])


In [6]:
y_pred.max()

tensor(15.9424)

In [7]:
#without fgmask
dice = DiceLoss(sigmoid=True)
dice(y_pred,y_true)


tensor(0.7414)

In [8]:
#without fgmask
bce = nn.BCEWithLogitsLoss()
bce(y_pred,y_true.float())

tensor(0.8966)

In [120]:
dice_loss = 0.0
bce_loss = 0.0
for c in range(3):
    pred_region = y_pred[:, c].unsqueeze(1)
    true_region = y_true[:, c].unsqueeze(1)
    dice_loss += dice(pred_region, true_region)
    bce_loss += bce(pred_region, true_region.float())

dice_loss /= 3
bce_loss /= 3

In [122]:
bce_loss

tensor(0.8973)

In [9]:
#mask preds with foreground first, then sigmoid
dice = DiceLoss(sigmoid=True)
dice(y_pred*fg_mask,y_true)

tensor(0.7415)

In [10]:
#mask preds with foreground first, then sigmoid
bce = nn.BCEWithLogitsLoss()
bce(y_pred*fg_mask,y_true.float())

tensor(0.7194)

In [12]:
#first sigmoid then mask preds with foreground
dice = DiceLoss(sigmoid=False)
dice(y_pred.sigmoid()*fg_mask, y_true)

tensor(0.5200)

In [114]:
bce = nn.BCELoss()
bce(y_pred.sigmoid()*fg_mask, y_true.float())

tensor(0.3188)

In [115]:
bce = nn.BCEWithLogitsLoss(weight=fg_mask)
bce(y_pred, y_true.float())

tensor(0.3188)